# Benchmark Results Analysis

Loads audited model outputs, computes accuracy, per-difficulty breakdown, correlation with human solve rate, and cross-model comparison. Includes F1/Precision/Recall per difficulty tier.

## 1. Imports

In [ ]:
import re
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import pearsonr
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

WORKDIR = './audit_outputs'
GT_PATH = '/content/puzzle_samples.xlsx'
VALID_ANSWERS = {'A', 'B', 'C', 'D', 'E'}

## 2. Load Data

In [ ]:
# Load combined audited results
combined = pd.read_csv(os.path.join('/content/all_models_audited.csv'))

In [ ]:
gemini = pd.read_csv("/content/all_models_audited_only_gemini.csv")

In [ ]:
combined.info()

In [ ]:
gemini.info()

In [ ]:
# Remove old Gemini benchmark rows
combined_no_gemini = combined[combined['model_name'] != 'gemini-2.5-flash']
print(f'Rows before: {len(combined)}')
print(f'Rows after removing Gemini: {len(combined_no_gemini)}')

In [ ]:
# Concat and save
combined_updated = pd.concat([combined_no_gemini, gemini], ignore_index=True)

In [ ]:
combined_updated.to_csv('all_models_audited_final.csv', index=False)


In [ ]:
combined = combined_updated

In [ ]:
# Load ground truth
if GT_PATH.endswith('.xlsx'):
    gt = pd.read_excel(GT_PATH)
else:
    gt = pd.read_csv(GT_PATH)

def map_difficulty(p):
    if p < 30:   return 'Hard'
    elif p < 60: return 'Medium'
    else:        return 'Easy'

if 'Solved_Percentile' not in combined.columns:
    combined = combined.merge(
        gt[['Puzzle_Number', 'Question_Number', 'Solved_Percentile']],
        on=['Puzzle_Number', 'Question_Number'], how='left'
    )

combined['difficulty'] = combined['Solved_Percentile'].apply(map_difficulty)

# UNKNOWN/NONE treated as wrong — do NOT filter them out
# A model that failed to answer is penalised in accuracy
combined['correct'] = combined['audited_answer'] == combined['GT_Answer']

models_list = sorted(combined['model_name'].unique())

print(f'Total rows    : {len(combined)}')
print(f'Models        : {models_list}')
print(f'\nDifficulty distribution:')
print(combined['difficulty'].value_counts().to_string())
print(f'\nAnswer coverage per model:')
for model_name, g in combined.groupby('model_name'):
    valid_n  = g['audited_answer'].isin(['A','B','C','D','E']).sum()
    total_n  = len(g)
    print(f'  {model_name:<30} {valid_n}/{total_n} valid ({valid_n/total_n:.1%})')


## 3. Per-Model Accuracy Table

In [ ]:
if not os.path.exists(WORKDIR):
    os.makedirs(WORKDIR)

In [ ]:
DIFFICULTY_ORDER = ['Hard', 'Medium', 'Easy']
VALID_ANSWERS    = {'A', 'B', 'C', 'D', 'E'}

def model_metrics(g: pd.DataFrame) -> dict:
    """Accuracy + F1/Precision/Recall. UNKNOWN/NONE count as wrong."""
    y_true = g['GT_Answer']
    y_pred = g['audited_answer']
    acc    = (y_true == y_pred).mean()

    # Only compute F1 on rows with valid predictions (UNKNOWN skews macro)
    valid_mask = y_pred.isin(VALID_ANSWERS)
    if valid_mask.sum() > 0:
        labels = list(VALID_ANSWERS)
        f1  = f1_score(y_true[valid_mask], y_pred[valid_mask], labels=labels, average='macro', zero_division=0)
        pr  = precision_score(y_true[valid_mask], y_pred[valid_mask], labels=labels, average='macro', zero_division=0)
        rec = recall_score(y_true[valid_mask], y_pred[valid_mask], labels=labels, average='macro', zero_division=0)
    else:
        f1 = pr = rec = 0.0

    coverage = valid_mask.mean()
    return {'acc': acc, 'f1': f1, 'precision': pr, 'recall': rec,
            'n': len(g), 'coverage': coverage}


rows = []
for model_name, g in combined.groupby('model_name'):
    m = model_metrics(g)
    row = {
        'model':      model_name,
        'n':          m['n'],
        'coverage_%': round(m['coverage'] * 100, 1),
        'acc_%':      round(m['acc'] * 100, 1),
        'f1':         round(m['f1'],        3),
        'precision':  round(m['precision'], 3),
        'recall':     round(m['recall'],    3),
    }
    for diff in DIFFICULTY_ORDER:
        sub = g[g['difficulty'] == diff]
        if len(sub):
            dm = model_metrics(sub)
            row[f'acc_{diff}_%'] = round(dm['acc'] * 100, 1)
            row[f'n_{diff}']     = dm['n']
        else:
            row[f'acc_{diff}_%'] = None
            row[f'n_{diff}']     = 0
    rows.append(row)

metrics_df = pd.DataFrame(rows).sort_values('acc_%', ascending=False)
metrics_df.to_csv(os.path.join(WORKDIR, 'model_metrics.csv'), index=False)

print('Cross-model metrics (UNKNOWN/NONE counted as wrong):')
print(metrics_df.to_string(index=False))


## 4. Accuracy vs Human Difficulty (Scatter + Correlation)

In [ ]:
BINS        = [0, 30, 60, 100]
LABELS      = ['Hard\n(<30)', 'Medium\n(30-60)', 'Easy\n(≥60)']
BIN_CENTERS = [15, 45, 80]

ncols = 3
nrows = -(-len(models_list) // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for idx, model_name in enumerate(models_list):
    ax = axes[idx]
    g  = combined[combined['model_name'] == model_name].copy()
    g['bin'] = pd.cut(g['Solved_Percentile'], bins=BINS, labels=LABELS, right=False)

    acc_by_bin = g.groupby('bin')['correct'].mean() * 100
    counts     = g.groupby('bin')['correct'].count()

    r, p = pearsonr(g['Solved_Percentile'], g['correct'].astype(int))
    p_str = 'p<0.001' if p < 0.001 else f'p={p:.3f}'

    ax.bar(BIN_CENTERS, acc_by_bin.reindex(LABELS).values,
           width=20, alpha=0.6, color=['#d62728','#ff7f0e','#2ca02c'],
           edgecolor='black', linewidth=0.5)
    ax.plot(BIN_CENTERS, acc_by_bin.reindex(LABELS).values,
            '-o', color='navy', linewidth=1.5, markersize=6, zorder=5)

    for xc, label in zip(BIN_CENTERS, LABELS):
        n_bin = counts.reindex(LABELS).get(label, 0)
        ax.text(xc, 2, f'n={n_bin}', ha='center', va='bottom', fontsize=8,
                color='white', fontweight='bold')

    ax.set_title(f'{model_name}\n(r={r:.3f}, {p_str})', fontsize=10, fontweight='bold')
    ax.set_xlabel('Human Solved Percentile (%)', fontsize=9)
    ax.set_ylabel('Model Accuracy (%)', fontsize=9)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_xticks([15, 45, 80])
    ax.set_xticklabels(['Hard\n<30', 'Medium\n30-60', 'Easy\n≥60'], fontsize=8)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.grid(alpha=0.3)

for ax in axes[len(models_list):]:
    ax.set_visible(False)

plt.suptitle('Model Accuracy vs Human Difficulty\n(UNKNOWN/NONE counted as wrong)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(WORKDIR, 'accuracy_vs_difficulty.png'), dpi=150, bbox_inches='tight')
plt.show()


## 5. Cross-Model Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Overall accuracy ──────────────────────────────────────────────────────────
ax = axes[0]
order = metrics_df.sort_values('acc_%', ascending=True)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(order)))
bars = ax.barh(order['model'], order['acc_%'], color=colors, edgecolor='black', linewidth=0.5)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xlabel('Accuracy (%)', fontweight='bold')
ax.set_title('Overall Accuracy by Model', fontweight='bold')
ax.set_xlim(0, 110)
ax.axvline(x=20, color='grey', linestyle='--', alpha=0.4, label='Random (20%)')
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.3)

# ── Accuracy by difficulty tier ───────────────────────────────────────────────
ax = axes[1]
x      = np.arange(len(metrics_df))
width  = 0.25
colors_diff = {'Hard': '#d62728', 'Medium': '#ff7f0e', 'Easy': '#2ca02c'}

for i, diff in enumerate(DIFFICULTY_ORDER):
    vals = metrics_df[f'acc_{diff}_%'].fillna(0)
    ax.bar(x + i * width, vals, width, label=diff,
           color=colors_diff[diff], alpha=0.75, edgecolor='black', linewidth=0.5)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_df['model'], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_title('Accuracy by Difficulty Tier', fontweight='bold')
ax.legend(title='Difficulty')
ax.set_ylim(0, 110)
ax.axhline(y=20, color='grey', linestyle='--', alpha=0.4)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(WORKDIR, 'cross_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()


## 6. F1 / Precision / Recall Discussion

> For a 5-class MCQ task, **accuracy is the primary metric** since all classes are equally valid answer choices and there's no meaningful notion of false positives being worse than false negatives. Macro F1 is reported for completeness and to catch any systematic bias toward certain answer letters, but it should track closely with accuracy if the model isn't degenerate. If macro F1 diverges significantly from accuracy for a model, it suggests the model is over-predicting certain answer letters — worth investigating in the confusion matrix below.

## 7. Confusion Matrix per Model (answer letter bias check)

In [ ]:
# Only use valid answers for confusion matrix (UNKNOWN/NONE can't go in a 5x5 matrix)
# But note: these rows are already penalised in accuracy above
valid_cm = combined[combined['audited_answer'].isin(VALID_ANSWERS)]

fig, axes = plt.subplots(
    -(-len(models_list) // 4), min(4, len(models_list)),
    figsize=(4 * min(4, len(models_list)), 3.5 * (-(-len(models_list) // 4)))
)
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
labels = sorted(VALID_ANSWERS)

for idx, model_name in enumerate(models_list):
    ax = axes[idx]
    g  = valid_cm[valid_cm['model_name'] == model_name]
    cm = confusion_matrix(g['GT_Answer'], g['audited_answer'], labels=labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_xticklabels(labels); ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

    # Show coverage in title
    total = len(combined[combined['model_name'] == model_name])
    cov   = len(g) / total * 100
    ax.set_title(f'{model_name}\ncoverage={cov:.0f}%', fontsize=9, fontweight='bold')

    for i in range(5):
        for j in range(5):
            ax.text(j, i, f'{cm_norm[i,j]:.2f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if cm_norm[i,j] > 0.5 else 'black')

for ax in axes[len(models_list):]:
    ax.set_visible(False)

plt.suptitle('Normalised Confusion Matrices\n(on valid answers only; coverage shown in title)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(WORKDIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()


## 8. Pearson Correlation Summary Table

In [ ]:
corr_rows = []
for model_name, g in combined.groupby('model_name'):
    r, p = pearsonr(g['Solved_Percentile'], g['correct'].astype(int))
    corr_rows.append({
        'model':       model_name,
        'pearson_r':   round(r, 3),
        'p_value':     round(p, 4),
        'significant': 'Yes' if p < 0.05 else 'No'
    })

corr_df = pd.DataFrame(corr_rows).sort_values('pearson_r', ascending=False)
corr_df.to_csv(os.path.join(WORKDIR, 'pearson_correlations.csv'), index=False)
print('Pearson r (accuracy vs human solve rate):')
print(corr_df.to_string(index=False))


## 9. All Models — Accuracy vs Difficulty (Combined)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

DIFF_ORDER  = ['Hard\n<30', 'Medium\n30-60', 'Easy\n≥60']
BIN_CENTERS = [15, 45, 80]
BINS_PLOT   = [0, 30, 60, 100]

cmap   = plt.cm.tab10
colors = {m: cmap(i / len(models_list)) for i, m in enumerate(sorted(models_list))}

for model_name, g in combined.groupby('model_name'):
    g = g.copy()
    g['bin'] = pd.cut(g['Solved_Percentile'], bins=BINS_PLOT,
                      labels=DIFF_ORDER, right=False)
    acc = g.groupby('bin')['correct'].mean() * 100
    vals = acc.reindex(DIFF_ORDER).values
    ax.plot(BIN_CENTERS, vals, 'o-', linewidth=2, markersize=7,
            label=model_name, color=colors[model_name])
    # Annotate endpoint
    ax.text(BIN_CENTERS[-1] + 1.5, vals[-1], f'{vals[-1]:.0f}%',
            va='center', fontsize=7.5, color=colors[model_name])

ax.set_xticks(BIN_CENTERS)
ax.set_xticklabels(DIFF_ORDER, fontsize=11)
ax.set_ylabel('Accuracy (%)', fontweight='bold', fontsize=11)
ax.set_xlabel('Difficulty Tier', fontweight='bold', fontsize=11)
ax.set_title('All Models: Accuracy vs Human Difficulty\n',
             fontweight='bold', fontsize=13)
ax.set_xlim(0, 105)
ax.set_ylim(0, 105)
ax.axhline(y=20, color='grey', linestyle='--', alpha=0.4, label='Random (20%)')
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left', framealpha=0.9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(WORKDIR, 'all_models_difficulty.png'), dpi=150, bbox_inches='tight')
plt.show()


# Original Vs Adapted Solve Rate

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ── Load both audited result files ────────────────────────────────────────────
original_path  = '/content/all_models_audited.csv'   # full 978 puzzle results
benchmark_path = '/content/all_models_audited_final.csv'  # 100 adapted puzzle results

orig  = pd.read_csv(original_path)
bench = pd.read_csv(benchmark_path)

# ── Merge on Puzzle_Number + Question_Number + model_name ─────────────────────
# Keep only puzzles that exist in both (the 100 benchmark puzzles)
merged = orig.merge(
    bench[['Puzzle_Number', 'Question_Number', 'model_name', 'correct', 'audited_answer']],
    on=['Puzzle_Number', 'Question_Number', 'model_name'],
    suffixes=('_orig', '_bench'),
    how='inner'
)

print(f'Matched rows: {len(merged)}')
print(f'Models: {sorted(merged["model_name"].unique())}')

# ── Per-model comparison table ────────────────────────────────────────────────
rows = []
for model_name, g in merged.groupby('model_name'):
    rows.append({
        'model':         model_name,
        'n':             len(g),
        'original_%':    round(g['correct_orig'].mean()  * 100, 1),
        'adapted_%':     round(g['correct_bench'].mean() * 100, 1),
        'delta':         round((g['correct_bench'].mean() - g['correct_orig'].mean()) * 100, 1),
    })

cmp = pd.DataFrame(rows).sort_values('original_%', ascending=False)
cmp['harder'] = cmp['delta'] < 0
print('\nOriginal vs Adapted Solve Rate (same 100 puzzles):')
print(cmp.to_string(index=False))
cmp.to_csv('original_vs_adapted_comparison.csv', index=False)

# ── Grouped bar chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

x     = np.arange(len(cmp))
width = 0.35

ax.bar(x - width/2, cmp['original_%'], width, label='Original',
       color='#5b8db8', edgecolor='#333333', linewidth=0.6, alpha=0.85)
ax.bar(x + width/2, cmp['adapted_%'],  width, label='Adapted',
       color='#f4a261', edgecolor='#333333', linewidth=0.6, alpha=0.85)

# Delta annotations
for i, (_, row) in enumerate(cmp.iterrows()):
    color = '#c0392b' if row['delta'] < 0 else '#27ae60'
    ax.text(i, max(row['original_%'], row['adapted_%']) + 1.5,
            f"{'+' if row['delta'] >= 0 else ''}{row['delta']:.1f}%",
            ha='center', fontsize=8, color=color, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(cmp['model'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_title('Solve Rate: Original vs Adapted Puzzles',
             fontweight='bold')
ax.set_ylim(0, 115)
ax.axhline(y=20, color='grey', linestyle='--', alpha=0.4, label='Random (20%)')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig('original_vs_adapted_solve_rate.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Dumbell Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sort by original performance
cmp_plot = cmp.sort_values("original_%", ascending=False).reset_index(drop=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})

fig, ax = plt.subplots(figsize=(5, 2.6))

# ACL-friendly colors
ORIGINAL = "#4C72B0"
ADAPTED  = "#DD8452"
CONNECT  = "#C8C8C8"

y = np.arange(len(cmp_plot))

# Connecting lines
for i, row in cmp_plot.iterrows():
    ax.plot(
        [row["original_%"], row["adapted_%"]],
        [i, i],
        color=CONNECT,
        lw=1.2,
        zorder=1,
    )

# Original (circle)
ax.scatter(
    cmp_plot["original_%"],
    y,
    s=42,
    marker="o",
    color=ORIGINAL,
    edgecolor="black",
    linewidth=0.4,
    label="Original",
    zorder=3,
)

# Adapted (square)
ax.scatter(
    cmp_plot["adapted_%"],
    y,
    s=42,
    marker="s",
    color=ADAPTED,
    edgecolor="black",
    linewidth=0.4,
    label="Adapted",
    zorder=3,
)

# Label only meaningful changes (>=2 percentage points)
for i, row in cmp_plot.iterrows():
    if abs(row["delta"]) >= 2:
        xpos = max(row["original_%"], row["adapted_%"]) + 1.0
        ax.text(
            xpos,
            i,
            f"{row['delta']:+.1f}",
            fontsize=7,
            va="center",
            ha="left",
            color="black",
        )

# Axes
ax.set_xlim(0, 102)
ax.set_xticks(np.arange(0, 101, 20))
ax.set_xlabel("Accuracy (%)")

ax.set_yticks(y)
ax.set_yticklabels(cmp_plot["model"])

# Highest model on top
ax.invert_yaxis()

# Grid
ax.grid(axis="x", color="#E6E6E6", linewidth=0.6)
ax.set_axisbelow(True)

# Clean frame
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Legend ABOVE the axes (never floating inside)
ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=2,
    frameon=False,
    handletextpad=0.4,
    columnspacing=1.2,
)

plt.tight_layout()

plt.savefig(
    "original_vs_adapted_dumbbell.pdf",
    dpi=300,
    bbox_inches="tight",
)

plt.savefig(
    "original_vs_adapted_dumbbell.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
import shutil

# Create the zip archive
shutil.make_archive(
    "/content/audit_outputs",   # output zip (without .zip)
    "zip",
    "/content/audit_outputs"    # folder to zip
)